# Brain Tumor MRI - Data Preprocessing & Splitting

## Objectives
1. **Data Splitting**: Divide dataset into Train (70%), Validation (15%), Test (15%)
2. **Image Resizing**: Standardize all images to 224×224 pixels
3. **Normalization**: Scale pixel values from [0, 255] to [0, 1]
4. **Data Augmentation**: Generate augmented versions to improve generalization

## Why These Steps?
- **Training (70%)**: The "textbook" the model learns from
- **Validation (15%)**: The "practice test" to tune hyperparameters
- **Test (15%)**: The "final exam" to evaluate on unseen data
- **Augmentation**: Teaches the model that tumors are tumors regardless of angle/position

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Setup Paths and Configuration

In [ ]:
ORIGINAL_DATASET_PATH = Path('Dataset/brain_tumor_dataset')
SPLIT_DATASET_PATH = Path('Dataset/split_data')

CLASSES = ['glioma', 'healthy', 'meningioma', 'pituitary']

IMG_SIZE = (224, 224)

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_SEED = 42

print("Configuration:")
print(f"  Source: {ORIGINAL_DATASET_PATH}")
print(f"  Destination: {SPLIT_DATASET_PATH}")
print(f"  Image Size: {IMG_SIZE}")
print(f"  Split Ratio: Train={TRAIN_RATIO}, Val={VAL_RATIO}, Test={TEST_RATIO}")
print(f"  Random Seed: {RANDOM_SEED}")

## 2. Load Dataset and Prepare File List

In [ ]:
all_files = []
class_counts = {}

for class_name in CLASSES:
    class_path = ORIGINAL_DATASET_PATH / class_name
    image_files = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png')) + list(class_path.glob('*.jpeg'))
    
    for img_path in image_files:
        all_files.append({
            'path': str(img_path),
            'class': class_name,
            'filename': img_path.name
        })
    
    class_counts[class_name] = len(image_files)

df = pd.DataFrame(all_files)

print("="*60)
print("DATASET SUMMARY")
print("="*60)
for class_name, count in class_counts.items():
    print(f"{class_name.capitalize():15s}: {count:5d} images")
print("="*60)
print(f"{'Total':15s}: {len(df):5d} images")
print("="*60)

print(f"\nDataFrame shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

## 3. Stratified Train/Validation/Test Split

Using stratified splitting to maintain class balance across all splits.

In [ ]:
train_df, temp_df = train_test_split(
    df, 
    test_size=(VAL_RATIO + TEST_RATIO), 
    stratify=df['class'], 
    random_state=RANDOM_SEED
)

val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=(1 - val_ratio_adjusted), 
    stratify=temp_df['class'], 
    random_state=RANDOM_SEED
)

print("="*60)
print("SPLIT SUMMARY")
print("="*60)
print(f"Training set:   {len(train_df):5d} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation set: {len(val_df):5d} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set:       {len(test_df):5d} images ({len(test_df)/len(df)*100:.1f}%)")
print(f"Total:          {len(df):5d} images")
print("="*60)

In [ ]:
print("\n" + "="*60)
print("CLASS DISTRIBUTION IN EACH SPLIT")
print("="*60)

splits = {'Train': train_df, 'Validation': val_df, 'Test': test_df}

for split_name, split_df in splits.items():
    print(f"\n{split_name}:")
    class_dist = split_df['class'].value_counts().sort_index()
    for class_name in CLASSES:
        count = class_dist.get(class_name, 0)
        percentage = (count / len(split_df)) * 100
        print(f"  {class_name.capitalize():15s}: {count:4d} ({percentage:5.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, (split_name, split_df) in enumerate(splits.items()):
    class_counts = split_df['class'].value_counts().reindex(CLASSES)
    axes[idx].bar(CLASSES, class_counts, color=colors, alpha=0.8, edgecolor='black')
    axes[idx].set_xlabel('Class', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Number of Images', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{split_name} Set Distribution', fontsize=13, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    for i, count in enumerate(class_counts):
        axes[idx].text(i, count + 10, str(count), ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Create Split Directory Structure

In [ ]:
if SPLIT_DATASET_PATH.exists():
    print(f"⚠️  Directory {SPLIT_DATASET_PATH} already exists.")
    response = input("Do you want to remove it and recreate? (yes/no): ")
    if response.lower() == 'yes':
        shutil.rmtree(SPLIT_DATASET_PATH)
        print("✓ Removed existing directory")
    else:
        print("⚠️  Keeping existing directory. Files may be overwritten.")

for split in ['train', 'val', 'test']:
    for class_name in CLASSES:
        split_class_path = SPLIT_DATASET_PATH / split / class_name
        split_class_path.mkdir(parents=True, exist_ok=True)

print("\n✓ Directory structure created:")
print(f"  {SPLIT_DATASET_PATH}/")
print(f"    ├── train/")
print(f"    │   ├── glioma/")
print(f"    │   ├── healthy/")
print(f"    │   ├── meningioma/")
print(f"    │   └── pituitary/")
print(f"    ├── val/")
print(f"    │   ├── glioma/")
print(f"    │   ├── healthy/")
print(f"    │   ├── meningioma/")
print(f"    │   └── pituitary/")
print(f"    └── test/")
print(f"        ├── glioma/")
print(f"        ├── healthy/")
print(f"        ├── meningioma/")
print(f"        └── pituitary/")

## 5. Copy and Preprocess Images

This step:
1. **Resizes** images to 224×224
2. **Converts** to RGB (for consistency)
3. **Saves** to appropriate split directories

In [ ]:
def preprocess_and_copy_images(df, split_name):
    """
    Preprocess and copy images to the split directory.
    
    Args:
        df: DataFrame containing image paths and classes
        split_name: 'train', 'val', or 'test'
    """
    print(f"\nProcessing {split_name.upper()} set...")
    
    success_count = 0
    error_count = 0
    
    for idx, row in df.iterrows():
        try:
            img = Image.open(row['path'])
            
            img = img.convert('RGB')
            
            img_resized = img.resize(IMG_SIZE, Image.LANCZOS)
            
            dest_path = SPLIT_DATASET_PATH / split_name / row['class'] / row['filename']
            img_resized.save(dest_path, 'JPEG', quality=95)
            
            success_count += 1
            
            if (success_count + error_count) % 500 == 0:
                print(f"  Processed: {success_count + error_count}/{len(df)} images...")
                
        except Exception as e:
            print(f"  Error processing {row['filename']}: {e}")
            error_count += 1
    
    print(f"  ✓ {split_name.upper()} set complete: {success_count} successful, {error_count} errors")
    return success_count, error_count

In [ ]:
print("="*60)
print("PREPROCESSING AND COPYING IMAGES")
print("="*60)
print(f"Resizing all images to {IMG_SIZE[0]}×{IMG_SIZE[1]}")
print(f"Converting to RGB format")
print("="*60)

train_success, train_errors = preprocess_and_copy_images(train_df, 'train')
val_success, val_errors = preprocess_and_copy_images(val_df, 'val')
test_success, test_errors = preprocess_and_copy_images(test_df, 'test')

print("\n" + "="*60)
print("PREPROCESSING SUMMARY")
print("="*60)
print(f"Train:      {train_success} images processed, {train_errors} errors")
print(f"Validation: {val_success} images processed, {val_errors} errors")
print(f"Test:       {test_success} images processed, {test_errors} errors")
print(f"Total:      {train_success + val_success + test_success} images processed")
print("="*60)

## 6. Verify Split Integrity

In [ ]:
print("="*60)
print("VERIFYING SPLIT INTEGRITY")
print("="*60)

for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()}:")
    total = 0
    for class_name in CLASSES:
        split_class_path = SPLIT_DATASET_PATH / split / class_name
        count = len(list(split_class_path.glob('*')))
        total += count
        print(f"  {class_name.capitalize():15s}: {count:4d} images")
    print(f"  {'Total':15s}: {total:4d} images")

In [ ]:
sample_image_path = next((SPLIT_DATASET_PATH / 'train' / CLASSES[0]).glob('*'))
sample_img = Image.open(sample_image_path)

print(f"\nSample image verification:")
print(f"  Path: {sample_image_path.name}")
print(f"  Size: {sample_img.size}")
print(f"  Mode: {sample_img.mode}")

if sample_img.size == IMG_SIZE and sample_img.mode == 'RGB':
    print("  ✓ Preprocessing successful!")
else:
    print("  ⚠️  Unexpected image properties")

## 7. Visualize Preprocessed Samples

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Preprocessed Training Images (224×224, RGB)', fontsize=16, fontweight='bold')

for idx, class_name in enumerate(CLASSES):
    class_path = SPLIT_DATASET_PATH / 'train' / class_name
    image_files = list(class_path.glob('*'))[:4]
    
    for col, img_path in enumerate(image_files):
        img = Image.open(img_path)
        axes[idx, col].imshow(img)
        axes[idx, col].axis('off')
        
        if col == 0:
            axes[idx, col].set_ylabel(class_name.capitalize(), fontsize=12, fontweight='bold', rotation=90, labelpad=10)

plt.tight_layout()
plt.show()

## 8. Data Augmentation Configuration

Setting up augmentation strategies for training to improve model generalization.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    
    print(f"TensorFlow version: {tf.__version__}")
    tensorflow_available = True
except ImportError:
    print("TensorFlow not installed. Install with: pip install tensorflow")
    tensorflow_available = False

In [ ]:
if tensorflow_available:
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        zoom_range=0.15,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest'
    )
    
    val_test_datagen = ImageDataGenerator(rescale=1./255)
    
    print("="*60)
    print("DATA AUGMENTATION CONFIGURATION")
    print("="*60)
    print("\nTraining Set Augmentation:")
    print("  ✓ Rescaling: [0, 255] → [0, 1]")
    print("  ✓ Rotation: ±20 degrees")
    print("  ✓ Width/Height Shift: ±10%")
    print("  ✓ Horizontal Flip: Yes")
    print("  ✓ Zoom: ±15%")
    print("  ✓ Brightness: 0.8-1.2x")
    print("  ✗ Vertical Flip: No (not anatomically meaningful)")
    print("\nValidation/Test Set:")
    print("  ✓ Rescaling only: [0, 255] → [0, 1]")
    print("  ✗ No augmentation (evaluate on original data)")
    print("="*60)
else:
    print("⚠️  TensorFlow not available. Install to use augmentation.")

## 9. Create Data Generators (Ready for Model Training)

In [ ]:
if tensorflow_available:
    BATCH_SIZE = 32
    
    train_generator = train_datagen.flow_from_directory(
        SPLIT_DATASET_PATH / 'train',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True,
        seed=RANDOM_SEED
    )
    
    val_generator = val_test_datagen.flow_from_directory(
        SPLIT_DATASET_PATH / 'val',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    
    test_generator = val_test_datagen.flow_from_directory(
        SPLIT_DATASET_PATH / 'test',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    
    print("\n✓ Data generators created successfully!")
    print(f"\nClass indices: {train_generator.class_indices}")
    print(f"\nBatch size: {BATCH_SIZE}")
    print(f"Train batches: {len(train_generator)}")
    print(f"Validation batches: {len(val_generator)}")
    print(f"Test batches: {len(test_generator)}")
else:
    print("⚠️  Cannot create generators without TensorFlow")

## 10. Visualize Augmented Images

In [ ]:
if tensorflow_available:
    sample_img_path = next((SPLIT_DATASET_PATH / 'train' / CLASSES[0]).glob('*'))
    sample_img = Image.open(sample_img_path)
    sample_array = np.array(sample_img).reshape((1,) + np.array(sample_img).shape)
    
    augmentation_generator = train_datagen.flow(
        sample_array, 
        batch_size=1,
        seed=RANDOM_SEED
    )
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Original vs Augmented Images (Examples)', fontsize=16, fontweight='bold')
    
    axes[0, 0].imshow(sample_img)
    axes[0, 0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    for i in range(1, 4):
        axes[0, i].axis('off')
    
    for i in range(4):
        augmented_img = next(augmentation_generator)[0].astype('uint8')
        axes[1, i].imshow(augmented_img)
        axes[1, i].set_title(f'Augmented {i+1}', fontsize=12, fontweight='bold')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Notice how augmentation creates variations:")
    print("   - Rotations at different angles")
    print("   - Horizontal flips")
    print("   - Slight zooms and shifts")
    print("   - Brightness variations")
    print("\n   These help the model learn that a tumor is still a tumor!")
else:
    print("⚠️  Cannot visualize augmentation without TensorFlow")

## 11. Save Split Information

In [ ]:
train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)
test_df.to_csv('test_split.csv', index=False)

print("✓ Split information saved:")
print("  - train_split.csv")
print("  - val_split.csv")
print("  - test_split.csv")

## Summary & Next Steps

### ✅ Completed Tasks

1. **Data Split**: Dataset divided into Train (70%), Validation (15%), Test (15%) with stratification
2. **Image Preprocessing**: All images resized to 224×224 and converted to RGB
3. **Normalization Setup**: Pixel values scaled from [0, 255] to [0, 1] via generators
4. **Data Augmentation**: Configured rotation, flip, zoom, brightness variations for training
5. **Generators Created**: Ready-to-use data generators for model training

### 📂 Output Structure
```
Dataset/split_data/
├── train/     (70% of data with augmentation)
├── val/       (15% of data, no augmentation)
└── test/      (15% of data, no augmentation)
```

### 🎯 Next Steps

1. **Model Architecture**: Build CNN or use transfer learning (VGG16, ResNet, EfficientNet)
2. **Training**: Train model using `train_generator` and validate with `val_generator`
3. **Evaluation**: Test on `test_generator` with accuracy, confusion matrix, classification report
4. **Optimization**: Fine-tune hyperparameters, try different architectures

### 💡 Key Points

- **Training set**: Learns patterns from augmented data
- **Validation set**: Monitors overfitting during training
- **Test set**: Final evaluation on completely unseen data
- **Normalization**: Helps model converge faster
- **Augmentation**: Prevents overfitting and improves generalization